# ACC Haufe reactivation + HPC theta phase organization

For each subject:

1. Project cue-period **ACC HFA** through that subject's **encoding-period Haufe activation pattern**
   to produce a scalar **reactivation trace** `(n_trials, n_cue_times)`. Haufe vectors are read from
   `ACC_rpe_category_hfa_200_posthoc_patterns.csv`. The active contrast (`positive_vs_neutral`
   or `positive_vs_negative`) is selected via the `SELECTED_CONTRAST` config knob.
2. Extract **HPC theta phase** (4-8 Hz, Hilbert) at the same cue-time samples for every HPC
   electrode in that subject.
3. Test per HPC electrode whether reactivation values are **more similar at more similar
   theta phases**: pairwise-sample Spearman between `react_sim = -|r_i - r_j|` and
   `phase_sim = cos(phi_i - phi_j)` on a fixed random subset of sample pairs.
4. Build a null by **per-trial circular shifts** of `reactivation` relative to `phi`, recomputing
   the RSA at the *same* sample pairs.
5. Aggregate per electrode -> per subject mean / best `z`. **Trial split** is set by `TRIAL_SPLIT`:
   **`hit_miss`** (default): contrast hit vs miss within subject where trial counts allow, plus
   group summaries. **`rpe_cat`**: split trials by RPE tercile (`neg`, `neutral`, `pos`) using
   `learn_df` aligned to `cue_on` epochs; per-category **n** is often small, so use a lower floor
   (`MIN_TRIALS_PER_RPE_CAT`) for per-subject point estimates and run **cross-subject** Friedman +
   pairwise Wilcoxon on subject-level means (see group stats cell).

**Output schema marker**: `RSA_PRIMITIVE` encodes the split (`split_hit_miss` vs `split_rpe_cat`)
so stale NPZs are invalidated when you change `TRIAL_SPLIT`.

Outputs land under `analysis_outputs/acc_hpc_phase_rsa/<SELECTED_CONTRAST>/<split_tag>/` so contrast
and trial-split combinations do not overwrite each other.


In [ ]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

import json
import math
import os
import warnings
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.colors import Normalize
import numpy as np
import pandas as pd
import seaborn as sns
from joblib import Parallel, delayed
from scipy.signal import hilbert, butter, filtfilt
from scipy.stats import friedmanchisquare, rankdata, spearmanr, ttest_rel, ttest_1samp, wilcoxon
from tqdm.auto import tqdm

import mne

from plot_config import set_plot_style
from mv_regression_utils import (
    Subject,
    _get_smoothed_event_signal,
    extract_theta_phase_from_cue_epochs,
)
from mv_granger_utils import build_hit_miss_trial_map, build_rpe_trial_map

warnings.filterwarnings('ignore')
set_plot_style()
sns.set_style('white')

# -----------------------------
# Paths
# -----------------------------
base_dir = '/Volumes/T7 Shield'
load_dir = f'{base_dir}/work/qasims01/MemoryBanditData/EMU'

HAUFE_PATTERNS_CSV: Path = Path(
    '/Volumes/T7 Shield/scratch/MemoryBandit/MVPA/'
    'decoding_feedback_posthoc_patterns/ACC_rpe_category_hfa_200_posthoc_patterns.csv'
)

# -----------------------------
# Active contrast
# -----------------------------
SELECTED_CONTRAST: str = 'positive_vs_neutral'   # or 'positive_vs_negative'
HAUFE_WEIGHT_COLUMN: str = 'PatternValue'        # 'PatternValue' (Haufe) or 'WeightValue' (raw)

# -----------------------------
# Trial split: hit/miss vs RPE terciles (separate output subfolders + cache marker)
# -----------------------------
TRIAL_SPLIT: str = 'hit_miss'   # 'hit_miss' | 'rpe_cat'
SPLIT_TAG: str = 'split_hit_miss' if TRIAL_SPLIT == 'hit_miss' else 'split_rpe_cat'
CONDITION_ORDER: Tuple[str, ...] = (
    ('hit', 'miss') if TRIAL_SPLIT == 'hit_miss' else ('neg', 'neutral', 'pos')
)

_CONTRAST_ROOT = Path(
    '/Users/salmanqasim/Documents/GitRepos/MemoryBandit_EMU/Notebooks/'
    'analysis_outputs/acc_hpc_phase_rsa'
) / SELECTED_CONTRAST
analysis_output_root = _CONTRAST_ROOT / SPLIT_TAG
per_subject_output_root = analysis_output_root / 'per_subject'
figure_output_root = analysis_output_root / 'figures'
for d in (per_subject_output_root, figure_output_root):
    d.mkdir(parents=True, exist_ok=True)

# -----------------------------
# HFA / theta config
# -----------------------------
ACC_ROI: str = 'ACC'
HPC_ROI: str = 'HPC'
EVENT_CUE: str = 'cue_on'

CUE_WINDOW: Tuple[float, float] = (0.0, 1.0)
THETA_BAND: Tuple[float, float] = (4.0, 8.0)

FREQ_RANGE: Tuple[float, float] = (70.0, 201.0)
DOWNSAMPLE_FACTOR: int = 5
WIN_MS: int = 200
HFA_SMOOTH_METHOD: str = 'movmean'
HFA_ARTIFACT_ABS_THRESH: float = 10.0

# -----------------------------
# Phase-similarity RSA config
# -----------------------------
N_PHASE_PERM: int = 500          # per-trial circular-shift surrogates per electrode
N_PAIR_SAMPLE: int = 50000       # random sample pairs used for the pairwise RSA
PHASE_SEED: int = 7

# -----------------------------
# Subject / electrode eligibility
# -----------------------------
MIN_TRIALS: int = 10
MIN_TRIALS_PER_COND: int = 8
MIN_TRIALS_PER_RPE_CAT: int = 5  # lower floor for rpe_cat point estimates only
MIN_PATTERN_ELECTRODES: int = 3
MIN_HPC_ELECTRODES: int = 1

# -----------------------------
# Parallelism
# -----------------------------
N_JOBS: int = -1
JOBLIB_BACKEND: str = 'loky'

# -----------------------------
# Cache schema marker (includes split tag)
# -----------------------------
RSA_PRIMITIVE: str = f'haufe_reactivation_phase_rsa_v2_{SPLIT_TAG}'

COND_SEED_CODE: Dict[str, int] = {
    'hit': 1, 'miss': 2, 'neg': 3, 'neutral': 4, 'pos': 5,
}

print('Output roots:')
for d in (per_subject_output_root, figure_output_root):
    print(f'  {d}')
print(f'HAUFE_PATTERNS_CSV    : {HAUFE_PATTERNS_CSV}')
print(f'SELECTED_CONTRAST     : {SELECTED_CONTRAST}')
print(f'TRIAL_SPLIT           : {TRIAL_SPLIT}  ({SPLIT_TAG})')
print(f'CONDITION_ORDER       : {CONDITION_ORDER}')
print(f'HAUFE_WEIGHT_COLUMN   : {HAUFE_WEIGHT_COLUMN}')
print(f'CUE_WINDOW            : {CUE_WINDOW}')
print(f'THETA_BAND            : {THETA_BAND}')
print(f'N_PHASE_PERM          : {N_PHASE_PERM}')
print(f'N_PAIR_SAMPLE         : {N_PAIR_SAMPLE}')
print(f'MIN_PATTERN_ELECTRODES: {MIN_PATTERN_ELECTRODES}')
print(f'MIN_TRIALS_PER_COND   : {MIN_TRIALS_PER_COND}')
print(f'MIN_TRIALS_PER_RPE_CAT: {MIN_TRIALS_PER_RPE_CAT}')


In [ ]:
# -----------------------------
# Inline helpers
# -----------------------------


def _window_mask(times: np.ndarray, window: Optional[Tuple[float, float]]) -> np.ndarray:
    if window is None:
        return np.ones_like(times, dtype=bool)
    t0, t1 = float(window[0]), float(window[1])
    return (times >= t0) & (times <= t1)


def load_haufe_patterns_table(csv_path: Path) -> pd.DataFrame:
    '''Read the posthoc patterns CSV. Returns a DataFrame with the original columns.'''
    df = pd.read_csv(csv_path)
    required = {'ROI', 'Contrast', 'Subject', 'Electrode', 'PatternValue', 'WeightValue'}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f'Patterns CSV missing required columns: {missing}')
    df['Subject'] = df['Subject'].astype(str)
    df['Electrode'] = df['Electrode'].astype(str)
    return df


def subject_acc_haufe_vector(
    patterns_df: pd.DataFrame,
    subject_id: str,
    contrast: str,
    electrode_labels: np.ndarray,
    *,
    value_col: str = 'PatternValue',
    min_kept: int = 3,
) -> Tuple[Optional[np.ndarray], Optional[np.ndarray], Optional[np.ndarray]]:
    '''Inner-join CSV electrodes with `electrode_labels`.

    Returns (w_subj, kept_labels, kept_indices) aligned to the cue-HFA electrode order.
    Returns (None, None, None) when fewer than `min_kept` electrodes match.
    '''
    sub = patterns_df[
        (patterns_df['ROI'] == 'ACC')
        & (patterns_df['Contrast'] == contrast)
        & (patterns_df['Subject'] == subject_id)
    ]
    if len(sub) == 0:
        return None, None, None
    pmap: Dict[str, float] = {}
    for lab, val in zip(sub['Electrode'].astype(str), sub[value_col].astype(float)):
        if np.isfinite(val):
            pmap[lab] = float(val)
    elec_array = np.asarray(electrode_labels, dtype=str)
    keep_mask = np.array([lbl in pmap for lbl in elec_array], dtype=bool)
    if int(keep_mask.sum()) < int(min_kept):
        return None, None, None
    kept_indices = np.where(keep_mask)[0].astype(np.int64)
    kept_labels = elec_array[kept_indices]
    w = np.asarray([pmap[lbl] for lbl in kept_labels], dtype=np.float64)
    return w, kept_labels, kept_indices


def compute_acc_reactivation(cue_hfa_3d: np.ndarray, w_subj: np.ndarray) -> np.ndarray:
    '''(n_trials, n_elec, n_t) x (n_elec,) -> (n_trials, n_t) scalar reactivation.'''
    return np.einsum(
        'tec,e->tc',
        np.asarray(cue_hfa_3d, dtype=np.float64),
        np.asarray(w_subj, dtype=np.float64),
    )


# ---- Phase-similarity RSA primitives ----

def _build_pair_indices(
    valid_flat_idx: np.ndarray,
    n_pair_sample: int,
    rng: np.random.Generator,
) -> Tuple[np.ndarray, np.ndarray]:
    '''Sample `n_pair_sample` (i, j) sample-index pairs, resolving self-pairs.'''
    n = valid_flat_idx.size
    if n < 20:
        return np.empty(0, dtype=np.int64), np.empty(0, dtype=np.int64)
    n_use = int(min(int(n_pair_sample), n * (n - 1)))
    i = valid_flat_idx[rng.integers(0, n, size=n_use)]
    j = valid_flat_idx[rng.integers(0, n, size=n_use)]
    self_mask = (i == j)
    while bool(self_mask.any()):
        k = np.where(self_mask)[0]
        j[k] = valid_flat_idx[rng.integers(0, n, size=k.size)]
        self_mask = (i == j)
    return i.astype(np.int64), j.astype(np.int64)


def _rho_at_pairs(r_flat: np.ndarray, p_flat: np.ndarray,
                  ii: np.ndarray, jj: np.ndarray) -> float:
    if ii.size == 0:
        return float('nan')
    react_sim = -np.abs(r_flat[ii] - r_flat[jj])
    phase_sim = np.cos(p_flat[ii] - p_flat[jj])
    if not (np.std(react_sim) > 0 and np.std(phase_sim) > 0):
        return float('nan')
    rho, _ = spearmanr(react_sim, phase_sim)
    return float(rho)


def phase_similarity_rsa_with_null(
    reactivation_2d: np.ndarray,
    phi_2d: np.ndarray,
    n_perm: int,
    n_pair_sample: int,
    seed: int,
) -> Dict[str, Any]:
    '''Sample-pair Spearman RSA + per-trial circular-shift null at the same pair set.'''
    r2 = np.asarray(reactivation_2d, dtype=np.float64)
    p2 = np.asarray(phi_2d, dtype=np.float64)
    if r2.shape != p2.shape:
        raise ValueError(f'shape mismatch reactivation={r2.shape} phi={p2.shape}')
    n_trials, n_t = r2.shape
    rng = np.random.default_rng(int(seed))

    r_flat = r2.ravel()
    p_flat = p2.ravel()
    valid_mask = np.isfinite(r_flat) & np.isfinite(p_flat)
    valid_flat_idx = np.where(valid_mask)[0]
    ii, jj = _build_pair_indices(valid_flat_idx, n_pair_sample, rng)
    if ii.size == 0:
        nan_arr = np.full(int(n_perm), np.nan, dtype=np.float64)
        return {
            'rho_obs': float('nan'),
            'z': float('nan'),
            'p': float('nan'),
            'rho_null_mean': float('nan'),
            'rho_null_std': float('nan'),
            'rho_null': nan_arr,
            'n_pairs': 0,
            'n_valid_samples': int(valid_flat_idx.size),
        }

    rho_obs = _rho_at_pairs(r_flat, p_flat, ii, jj)
    null = np.full(int(n_perm), np.nan, dtype=np.float64)
    if n_t < 2:
        rho_null_mean = float('nan')
        rho_null_std = 1e-12
    else:
        shifts = rng.integers(1, n_t, size=(int(n_perm), int(n_trials)))
        for k in range(int(n_perm)):
            r_sh = np.empty_like(r2)
            for ti in range(n_trials):
                r_sh[ti, :] = np.roll(r2[ti, :], int(shifts[k, ti]))
            null[k] = _rho_at_pairs(r_sh.ravel(), p_flat, ii, jj)
        rho_null_mean = float(np.nanmean(null)) if np.any(np.isfinite(null)) else float('nan')
        rho_null_std = max(
            float(np.nanstd(null, ddof=1)) if np.any(np.isfinite(null)) else 0.0,
            1e-12,
        )
    z = float((rho_obs - rho_null_mean) / rho_null_std) if np.isfinite(rho_null_mean) else float('nan')
    # one-sided permutation p-value (rho_obs >= null)
    finite_null = null[np.isfinite(null)]
    if finite_null.size == 0 or not np.isfinite(rho_obs):
        p_val = float('nan')
    else:
        p_val = float((np.sum(finite_null >= rho_obs) + 1) / (finite_null.size + 1))
    return {
        'rho_obs': float(rho_obs),
        'z': z,
        'p': p_val,
        'rho_null_mean': rho_null_mean,
        'rho_null_std': rho_null_std,
        'rho_null': null,
        'n_pairs': int(ii.size),
        'n_valid_samples': int(valid_flat_idx.size),
    }


# ---- Supplementary Moore-Rayleigh diagnostic (re-used from older runs) ----

def weighted_rayleigh(phi: np.ndarray, w: np.ndarray,
                      method: str = 'moore') -> Tuple[float, float]:
    '''Resultant length R and preferred phase, weighted by `w`.

    method='standard': R = |sum(w * exp(j*phi))| / sum(|w|)
    method='moore'   : R = |sum(rank(|w|) * sign(w) * exp(j*phi))| / sum(rank(|w|))
    '''
    phi = np.asarray(phi, dtype=np.float64).reshape(-1)
    w = np.asarray(w, dtype=np.float64).reshape(-1)
    mask = np.isfinite(phi) & np.isfinite(w)
    if mask.sum() < 5:
        return float('nan'), float('nan')
    phi = phi[mask]
    w = w[mask]
    if method == 'moore':
        ranks = rankdata(np.abs(w), method='average')
        weights = ranks * np.sign(w)
    elif method == 'standard':
        weights = w
    else:
        raise ValueError(method)
    denom = float(np.sum(np.abs(weights)))
    if denom <= 0:
        return float('nan'), float('nan')
    v = float(np.sum(weights * np.cos(phi))) + 1j * float(np.sum(weights * np.sin(phi)))
    return float(np.abs(v) / denom), float(np.angle(v))


# ---- I/O / cache helpers ----

def _npz_is_current(npz_path: Path) -> bool:
    '''Return True iff a per-subject NPZ exists with the current RSA_PRIMITIVE marker.'''
    if not npz_path.exists():
        return False
    try:
        d = np.load(npz_path, allow_pickle=True)
        if 'RSA_PRIMITIVE' not in d.files:
            return False
        if str(d['RSA_PRIMITIVE']) != RSA_PRIMITIVE:
            return False
        if 'SELECTED_CONTRAST' not in d.files:
            return False
        if str(d['SELECTED_CONTRAST']) != SELECTED_CONTRAST:
            return False
        if 'TRIAL_SPLIT' in d.files and str(d['TRIAL_SPLIT']) != TRIAL_SPLIT:
            return False
        return True
    except Exception:
        return False


In [ ]:
# -----------------------------
# Subject list
# -----------------------------
demo_file = f'{base_dir}/projects/guLab/Salman/EMU/MemoryBanditsDatabase.xlsx'
demo_data = pd.read_excel(demo_file, engine='openpyxl')
subj_df = demo_data[(demo_data.LongVersion == 1) & (demo_data.Minerva == 1)].copy()
subj_df = subj_df[~subj_df['MSSMCode'].isin(['MS015', 'MS024', 'MS034', 'MS038'])].copy()
subjects = sorted(subj_df['MSSMCode'].dropna().astype(str).unique().tolist())

haufe_patterns_df = load_haufe_patterns_table(HAUFE_PATTERNS_CSV)
csv_subjects = sorted(set(haufe_patterns_df['Subject'].astype(str)))
subjects_with_patterns = [s for s in subjects if s in set(csv_subjects)]

print('Eligible subjects total       :', len(subjects))
print('Subjects with Haufe patterns  :', len(subjects_with_patterns))
print('Sample subjects with patterns :', subjects_with_patterns[:10])

# Quick contrast summary
ctab = (
    haufe_patterns_df.groupby(['Contrast', 'Subject'])['Electrode']
    .nunique()
    .reset_index()
    .rename(columns={'Electrode': 'n_elec'})
)
print()
print('Contrast x Subject electrode counts (head):')
print(ctab.head(12).to_string(index=False))


In [ ]:
# -----------------------------
# Per-subject runner
# -----------------------------


def _build_subject(subject_id: str) -> Subject:
    return Subject(subject=subject_id, base_dir=base_dir, load_dir=load_dir,
                   min_trials=MIN_TRIALS, verbose=False)


def _load_acc_cue_hfa(subj: Subject) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    signal, times, elec_labels = _get_smoothed_event_signal(
        subject_obj=subj, roi=ACC_ROI, event=EVENT_CUE,
        freq_range=FREQ_RANGE, downsample_factor=DOWNSAMPLE_FACTOR,
        win_ms=WIN_MS, smooth_method=HFA_SMOOTH_METHOD,
        artifact_abs_thresh=HFA_ARTIFACT_ABS_THRESH,
    )
    times = np.asarray(times, dtype=np.float64)
    mask_t = _window_mask(times, CUE_WINDOW)
    sig = np.asarray(signal[..., mask_t], dtype=np.float64)
    return sig, times[mask_t], np.asarray(elec_labels, dtype=str)


def _n_epochs_cue_on(subj: Subject) -> int:
    epochs_file = os.path.join(
        subj.base_dir, 'projects', 'guLab', 'Salman', 'EphysAnalyses',
        subj.subject, 'neural', 'Day1', 'cue_on-epo.fif',
    )
    if not os.path.exists(epochs_file):
        raise FileNotFoundError(epochs_file)
    epochs = mne.read_epochs(epochs_file, preload=False, verbose=False)
    return int(len(epochs))


def _load_hit_miss_for_cue(subj: Subject) -> pd.DataFrame:
    n_epochs = _n_epochs_cue_on(subj)
    subj._ensure_behavior_loaded()
    return build_hit_miss_trial_map(
        subj._combined_df, subj_id=subj.subject,
        n_epochs=n_epochs, event='cue_on',
    )


def _load_rpe_cat_for_cue(subj: Subject) -> pd.DataFrame:
    n_epochs = _n_epochs_cue_on(subj)
    subj._ensure_behavior_loaded()
    return build_rpe_trial_map(
        subj.learn_df, subj_id=subj.subject,
        n_epochs=n_epochs, event='cue_on',
    )


def run_subject_phase_rsa(subject_id: str, overwrite: bool = False) -> Dict[str, Any]:
    out_npz = per_subject_output_root / f'{subject_id}_haufe_phase_rsa.npz'
    if (not overwrite) and _npz_is_current(out_npz):
        return {'subject': subject_id, 'status': 'cached', 'npz_path': str(out_npz)}

    try:
        subj = _build_subject(subject_id)
    except Exception as exc:
        return {'subject': subject_id, 'status': 'fail', 'reason': f'subject init: {exc!r}'}

    # ACC cue HFA
    try:
        acc_cue_hfa, cue_times, acc_labels = _load_acc_cue_hfa(subj)
    except Exception as exc:
        return {'subject': subject_id, 'status': 'fail', 'reason': f'load ACC HFA: {exc!r}'}

    w_subj, kept_acc_labels, kept_acc_idx = subject_acc_haufe_vector(
        haufe_patterns_df, subject_id, SELECTED_CONTRAST, acc_labels,
        value_col=HAUFE_WEIGHT_COLUMN, min_kept=MIN_PATTERN_ELECTRODES,
    )
    if w_subj is None:
        return {'subject': subject_id, 'status': 'skip',
                'reason': f'<{MIN_PATTERN_ELECTRODES} matched Haufe electrodes'}
    acc_cue_hfa = acc_cue_hfa[:, kept_acc_idx, :]

    min_per = MIN_TRIALS_PER_COND if TRIAL_SPLIT == 'hit_miss' else MIN_TRIALS_PER_RPE_CAT

    # Trial-condition map
    try:
        if TRIAL_SPLIT == 'hit_miss':
            beh_map = _load_hit_miss_for_cue(subj)
            cat_col = 'hit_miss'
            cond_indices = {
                'hit': np.where(beh_map[cat_col].astype(str) == 'hit')[0].astype(np.int64),
                'miss': np.where(beh_map[cat_col].astype(str) == 'miss')[0].astype(np.int64),
            }
        else:
            beh_map = _load_rpe_cat_for_cue(subj)
            cat_col = 'rpe_cat'
            cond_indices = {
                'neg': np.where(beh_map[cat_col].astype(str) == 'neg')[0].astype(np.int64),
                'neutral': np.where(beh_map[cat_col].astype(str) == 'neutral')[0].astype(np.int64),
                'pos': np.where(beh_map[cat_col].astype(str) == 'pos')[0].astype(np.int64),
            }
        n_epochs_beh = len(beh_map)
    except Exception as exc:
        return {'subject': subject_id, 'status': 'fail', 'reason': f'behavior map: {exc!r}'}
    if n_epochs_beh != acc_cue_hfa.shape[0]:
        return {'subject': subject_id, 'status': 'fail',
                'reason': f'epoch count mismatch behavior={n_epochs_beh} '
                          f'hfa={acc_cue_hfa.shape[0]}'}

    # Reactivation
    reactivation = compute_acc_reactivation(acc_cue_hfa, w_subj)
    n_trials, n_cue_times = reactivation.shape

    # HPC theta phase
    try:
        ph_out = extract_theta_phase_from_cue_epochs(
            subject_obj=subj, roi=HPC_ROI, event=EVENT_CUE,
            freq_band=THETA_BAND,
            target_times=cue_times.astype(np.float32),
        )
    except Exception as exc:
        return {'subject': subject_id, 'status': 'fail', 'reason': f'load HPC phase: {exc!r}'}
    phi_etc = np.asarray(ph_out['phase'], dtype=np.float64)         # (n_elec, n_t, n_trials)
    phi_per_elec = np.transpose(phi_etc, (2, 0, 1))                 # (n_trials, n_elec, n_t)
    hpc_labels = np.asarray(ph_out['electrode_labels'], dtype=str)
    if hpc_labels.size < MIN_HPC_ELECTRODES:
        return {'subject': subject_id, 'status': 'skip',
                'reason': f'<{MIN_HPC_ELECTRODES} HPC electrodes'}

    # Per-electrode phase-similarity RSA per condition
    per_elec: Dict[str, Dict[str, Any]] = {}
    for cond, trial_idx in cond_indices.items():
        if trial_idx.size < min_per:
            per_elec[cond] = {
                'status': 'skip',
                'reason': f'n_trials={trial_idx.size} < {min_per}',
            }
            continue
        n_h = int(hpc_labels.size)
        rho_obs = np.full(n_h, np.nan, dtype=np.float64)
        z_per = np.full(n_h, np.nan, dtype=np.float64)
        p_per = np.full(n_h, np.nan, dtype=np.float64)
        rho_null_mean = np.full(n_h, np.nan, dtype=np.float64)
        rho_null_std = np.full(n_h, np.nan, dtype=np.float64)
        react_cond = reactivation[trial_idx, :]
        scode = int(COND_SEED_CODE.get(cond, 0))
        for ei in range(n_h):
            phi_ei = phi_per_elec[trial_idx, ei, :]
            res = phase_similarity_rsa_with_null(
                react_cond, phi_ei,
                n_perm=N_PHASE_PERM,
                n_pair_sample=N_PAIR_SAMPLE,
                seed=PHASE_SEED + 101 * ei + scode,
            )
            rho_obs[ei] = res['rho_obs']
            z_per[ei] = res['z']
            p_per[ei] = res['p']
            rho_null_mean[ei] = res['rho_null_mean']
            rho_null_std[ei] = res['rho_null_std']
        per_elec[cond] = {
            'status': 'ok',
            'rho_obs': rho_obs,
            'z': z_per,
            'p': p_per,
            'rho_null_mean': rho_null_mean,
            'rho_null_std': rho_null_std,
            'n_trials': int(trial_idx.size),
        }

    payload: Dict[str, Any] = {
        'RSA_PRIMITIVE': RSA_PRIMITIVE,
        'TRIAL_SPLIT': np.asarray(TRIAL_SPLIT, dtype=object),
        'SELECTED_CONTRAST': SELECTED_CONTRAST,
        'subject': subject_id,
        'cue_times': cue_times.astype(np.float64),
        'reactivation': reactivation.astype(np.float32),
        'phi_per_elec': phi_per_elec.astype(np.float32),
        'acc_electrode_labels': kept_acc_labels.astype(object),
        'hpc_electrode_labels': hpc_labels.astype(object),
        'w_subj': w_subj.astype(np.float64),
        'n_trials': np.int64(n_trials),
        'n_pair_sample': np.int64(N_PAIR_SAMPLE),
        'n_phase_perm': np.int64(N_PHASE_PERM),
    }
    for cond in CONDITION_ORDER:
        payload[f'{cond}_indices'] = cond_indices[cond].astype(np.int64)
        payload[f'n_{cond}'] = np.int64(cond_indices[cond].size)

    for cond in CONDITION_ORDER:
        entry = per_elec.get(cond, {})
        status = entry.get('status', 'skip')
        payload[f'{cond}_status'] = np.asarray(status, dtype=object)
        if status != 'ok':
            payload[f'{cond}_reason'] = np.asarray(entry.get('reason', ''), dtype=object)
            continue
        for key in ('rho_obs', 'z', 'p', 'rho_null_mean', 'rho_null_std'):
            payload[f'{cond}_{key}'] = entry[key].astype(np.float64)
        payload[f'{cond}_n_trials'] = np.int64(entry['n_trials'])

    np.savez(out_npz, **payload)
    ret: Dict[str, Any] = {
        'subject': subject_id, 'status': 'ok', 'npz_path': str(out_npz),
        'reactivation': reactivation, 'cue_times': cue_times,
        'phi_per_elec': phi_per_elec, 'acc_electrode_labels': kept_acc_labels,
        'hpc_electrode_labels': hpc_labels, 'w_subj': w_subj,
        'cond_indices': cond_indices, 'per_elec': per_elec,
        'n_trials': int(n_trials), 'trial_split': TRIAL_SPLIT,
    }
    for cond in CONDITION_ORDER:
        ret[f'n_{cond}'] = int(cond_indices[cond].size)
    return ret


def load_subject_npz(subject_id: str) -> Optional[Dict[str, Any]]:
    out_npz = per_subject_output_root / f'{subject_id}_haufe_phase_rsa.npz'
    if not _npz_is_current(out_npz):
        return None
    d = dict(np.load(out_npz, allow_pickle=True))
    return d


def npz_condition_order(d: Dict[str, Any]) -> Tuple[str, ...]:
    ts = str(d.get('TRIAL_SPLIT', 'hit_miss'))
    if ts == 'rpe_cat':
        return ('neg', 'neutral', 'pos')
    return ('hit', 'miss')


## Single-subject demo

Run the pipeline on the first eligible subject end-to-end and print the per-electrode summary.
Behavior depends on `TRIAL_SPLIT` (`hit_miss` vs `rpe_cat`); outputs go under the matching `split_*` folder.


In [ ]:
# -----------------------------
# Single-subject demo
# -----------------------------
if not subjects_with_patterns:
    raise RuntimeError(
        f'No subjects have any rows in {HAUFE_PATTERNS_CSV} for ROI=ACC, '
        f'Contrast={SELECTED_CONTRAST!r}'
    )
DEMO_SUBJECT = subjects_with_patterns[0]
print(f'Running demo on {DEMO_SUBJECT} (TRIAL_SPLIT={TRIAL_SPLIT}) ...')
demo_res = run_subject_phase_rsa(DEMO_SUBJECT, overwrite=True)
print('Status:', demo_res.get('status'))

if demo_res.get('status') == 'ok':
    parts = [f"  n_trials={demo_res['n_trials']}"]
    for cond in CONDITION_ORDER:
        parts.append(f"n_{cond}={demo_res.get(f'n_{cond}', 'n/a')}")
    print('  ' + '  '.join(parts))
    print(f"  n_acc_pattern_electrodes={demo_res['acc_electrode_labels'].size}")
    print(f"  n_hpc_phase_electrodes  ={demo_res['hpc_electrode_labels'].size}")
    for cond in CONDITION_ORDER:
        entry = demo_res['per_elec'].get(cond, {})
        if entry.get('status') != 'ok':
            print(f"  {cond}: SKIP -- {entry.get('reason', '')}")
            continue
        mean_z = float(np.nanmean(entry['z']))
        best_i = int(np.nanargmax(entry['z']))
        best_z = float(entry['z'][best_i])
        best_lbl = str(demo_res['hpc_electrode_labels'][best_i])
        mean_p = float(np.nanmean(entry['p']))
        print(f"  {cond:>7}: n_trials={entry['n_trials']}  "
              f"mean_z={mean_z:+.3f}  best_z={best_z:+.3f} ({best_lbl})  "
              f"mean_p={mean_p:.3f}")
        top = np.argsort(-entry['z'])[:5]
        for ei in top:
            print(f"    elec[{ei}] {str(demo_res['hpc_electrode_labels'][ei]):>14}: "
                  f"rho_obs={entry['rho_obs'][ei]:+.3f}  "
                  f"z={entry['z'][ei]:+.2f}  p={entry['p'][ei]:.3g}")

    if TRIAL_SPLIT == 'hit_miss':
        # Paired hit vs miss: full per-electrode table + within-subject Wilcoxon on z
        hit_e = demo_res['per_elec'].get('hit', {})
        miss_e = demo_res['per_elec'].get('miss', {})
        if hit_e.get('status') == 'ok' and miss_e.get('status') == 'ok':
            labels = np.asarray(demo_res['hpc_electrode_labels'], dtype=object)
            hit_rho = np.asarray(hit_e['rho_obs'], dtype=np.float64)
            hit_z = np.asarray(hit_e['z'], dtype=np.float64)
            hit_p = np.asarray(hit_e['p'], dtype=np.float64)
            miss_rho = np.asarray(miss_e['rho_obs'], dtype=np.float64)
            miss_z = np.asarray(miss_e['z'], dtype=np.float64)
            miss_p = np.asarray(miss_e['p'], dtype=np.float64)
            paired_df = pd.DataFrame({
                'electrode': [str(x) for x in labels],
                'hit_rho': hit_rho,
                'hit_z': hit_z,
                'hit_p': hit_p,
                'miss_rho': miss_rho,
                'miss_z': miss_z,
                'miss_p': miss_p,
            })
            paired_df['delta_z_miss_minus_hit'] = paired_df['miss_z'] - paired_df['hit_z']
            print()
            print('Paired hit vs miss phase-RSA (per HPC electrode, delta_z = miss_z - hit_z):')
            print(paired_df.to_string(index=False, formatters={
                'hit_rho': lambda v: f'{v:+.4f}',
                'hit_z': lambda v: f'{v:+.4f}',
                'hit_p': lambda v: f'{v:.4g}',
                'miss_rho': lambda v: f'{v:+.4f}',
                'miss_z': lambda v: f'{v:+.4f}',
                'miss_p': lambda v: f'{v:.4g}',
                'delta_z_miss_minus_hit': lambda v: f'{v:+.4f}',
            }))
            fin = np.isfinite(hit_z) & np.isfinite(miss_z)
            n_fin = int(fin.sum())
            if n_fin >= 5:
                try:
                    w_stat, w_p = wilcoxon(
                        hit_z[fin], miss_z[fin], alternative='two-sided',
                    )
                    print(
                        'Within-subject paired Wilcoxon on z (hit vs miss, same electrode): '
                        f'n={n_fin}  statistic={float(w_stat):.4g}  two-sided p={float(w_p):.4g}'
                    )
                except ValueError as exc:
                    print(f'Within-subject paired Wilcoxon failed: {exc}')
            else:
                print(
                    'Within-subject paired Wilcoxon skipped: '
                    f'only {n_fin} finite paired z rows (need >= 5).'
                )
        else:
            print()
            print(
                'Paired hit vs miss table skipped: need per_elec hit and miss both status == ok.'
            )
    else:
        print()
        print(
            'TRIAL_SPLIT=rpe_cat: primary inference is cross-subject (Friedman / pairwise '
            'Wilcoxon on subject means) in the group statistics cell; per-category n is often '
            'too small for stable within-subject tests.'
        )


## Example-level intuition check

For the best-`z` HPC electrode in the **first** condition of `CONDITION_ORDER` with `status==ok`
from the single-subject demo (for default `TRIAL_SPLIT="hit_miss"` this is **hit**), render:

1. One example trial: theta-filtered HPC LFP colored by phase, an electrodes-by-time ACC HFA
   heatmap, and that trial's ACC reactivation trace across cue time.
2. Phase-vs-reactivation scatter pooled across that condition's (trial x time) samples.
3. Polar histogram of reactivation-weighted phase mass with the Moore-Rayleigh preferred phase
   overlaid for hit and miss.

The next code cell **prints** phase-RSA (`rho_obs`, permutation `z`, `p`) and Moore–Rayleigh summaries for **hit vs miss** on the best hit-`z` electrode before generating the figures.

**Note:** When `TRIAL_SPLIT="rpe_cat"`, the deep-dive cell is skipped (plots are hit/miss-specific).


In [ ]:
# -----------------------------
# Deep-dive example trial / electrode
# -----------------------------
if TRIAL_SPLIT != 'hit_miss':
    print(
        'Deep-dive intuition plots are only defined for TRIAL_SPLIT="hit_miss". '
        'Set TRIAL_SPLIT to "hit_miss" in the config cell and re-run the demo.'
    )
elif demo_res.get('status') == 'ok' and \
        demo_res['per_elec'].get('hit', {}).get('status') == 'ok':
    hit_entry = demo_res['per_elec']['hit']
    hpc_labels = demo_res['hpc_electrode_labels']
    best_ei = int(np.nanargmax(hit_entry['z']))
    example_hpc_label = str(hpc_labels[best_ei])
    cue_times = np.asarray(demo_res['cue_times'], dtype=np.float64)
    reactivation = np.asarray(demo_res['reactivation'], dtype=np.float64)
    phi_per_elec = np.asarray(demo_res['phi_per_elec'], dtype=np.float64)
    hit_idx = demo_res['cond_indices']['hit']
    miss_idx = demo_res['cond_indices']['miss']
    miss_entry = demo_res['per_elec'].get('miss', {})

    print()
    print(
        f'--- Best hit-z HPC electrode ei={best_ei} ({example_hpc_label}): '
        'RSA + Moore–Rayleigh (pooled trial x time) ---'
    )
    hdr = (
        f'{"cond":>5}  {"rho_obs":>9}  {"z":>9}  {"p":>10}  '
        f'{"Moore_R":>9}  {"pref_deg":>10}'
    )
    print(hdr)
    for cond, idx, ent in (
        ('hit', hit_idx, hit_entry),
        ('miss', miss_idx, miss_entry),
    ):
        if ent.get('status') != 'ok':
            print(f'{cond:>5}  (per_elec status not ok)')
            continue
        phi = phi_per_elec[idx, best_ei, :].reshape(-1)
        r = reactivation[idx, :].reshape(-1)
        mask = np.isfinite(phi) & np.isfinite(r)
        if mask.sum() == 0:
            R_m, pref_m = np.nan, np.nan
        else:
            R_m, pref_m = weighted_rayleigh(phi[mask], r[mask], method='moore')
        rho = float(ent['rho_obs'][best_ei])
        zv = float(ent['z'][best_ei])
        pv = float(ent['p'][best_ei])
        pref_deg = float(np.degrees(pref_m)) if np.isfinite(pref_m) else float('nan')
        print(
            f'{cond:>5}  {rho:+.4f}  {zv:+.4f}  {pv:10.4g}  '
            f'{R_m:9.4f}  {pref_deg:+10.1f}'
        )
    if miss_entry.get('status') == 'ok' and hit_entry.get('status') == 'ok':
        dz = float(miss_entry['z'][best_ei] - hit_entry['z'][best_ei])
        print(f'delta_z (miss - hit) at this electrode: {dz:+.4f}')

    # ---- Pick an example trial: highest |reactivation| on that electrode for hit ----
    react_hit = reactivation[hit_idx, :]
    trial_rank_local = int(np.nanargmax(np.nanmax(np.abs(react_hit), axis=1)))
    example_epoch_idx = int(hit_idx[trial_rank_local])
    example_react = reactivation[example_epoch_idx, :]

    # ---- Load HPC theta-filtered LFP at the example electrode for this trial ----
    subj_demo = _build_subject(DEMO_SUBJECT)
    epochs_file = os.path.join(
        subj_demo.base_dir, 'projects', 'guLab', 'Salman', 'EphysAnalyses',
        subj_demo.subject, 'neural', 'Day1', 'cue_on-epo.fif',
    )
    epochs = mne.read_epochs(epochs_file, preload=True, verbose=False)
    if example_hpc_label not in epochs.ch_names:
        for cand in epochs.ch_names:
            if cand.strip() == example_hpc_label.strip():
                example_hpc_label = cand
                break
    epochs_elec = epochs.copy().pick([example_hpc_label]).crop(*CUE_WINDOW)
    sfreq = float(epochs_elec.info['sfreq'])
    nyq = sfreq / 2.0
    b, a = butter(4, [THETA_BAND[0] / nyq, THETA_BAND[1] / nyq], btype='bandpass')
    raw_trial = epochs_elec.get_data(picks=0)[example_epoch_idx].ravel()
    theta_lfp = filtfilt(b, a, raw_trial)
    theta_phase = np.angle(hilbert(theta_lfp))
    theta_times = epochs_elec.times

    # ---- ACC HFA heatmap for the same trial ----
    acc_cue_hfa, _, _ = _load_acc_cue_hfa(subj_demo)
    w_subj, kept_acc_labels, kept_acc_idx = subject_acc_haufe_vector(
        haufe_patterns_df, DEMO_SUBJECT, SELECTED_CONTRAST,
        np.asarray(_load_acc_cue_hfa(subj_demo)[2], dtype=str),
        value_col=HAUFE_WEIGHT_COLUMN, min_kept=MIN_PATTERN_ELECTRODES,
    )
    acc_trial = acc_cue_hfa[example_epoch_idx][kept_acc_idx]

    fig, axes = plt.subplots(3, 1, figsize=(10.0, 7.0), sharex=False,
                             gridspec_kw={'height_ratios': [1.0, 1.1, 0.9]})

    # (1) Theta-filtered LFP colored by phase
    points = np.array([theta_times, theta_lfp]).T.reshape(-1, 1, 2)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)
    lc = LineCollection(segments, cmap='twilight', norm=Normalize(-np.pi, np.pi))
    lc.set_array(theta_phase[:-1])
    lc.set_linewidth(2.0)
    axes[0].add_collection(lc)
    axes[0].set_xlim(theta_times.min(), theta_times.max())
    ypad = 0.1 * np.nanmax(np.abs(theta_lfp)) if np.nanmax(np.abs(theta_lfp)) > 0 else 1.0
    axes[0].set_ylim(theta_lfp.min() - ypad, theta_lfp.max() + ypad)
    axes[0].set_ylabel('theta LFP')
    axes[0].set_title(f'{DEMO_SUBJECT} hit trial {example_epoch_idx}: '
                      f'{example_hpc_label} theta-filtered LFP colored by phase')
    cbar = fig.colorbar(lc, ax=axes[0], pad=0.01)
    cbar.set_label('phase (rad)')

    # (2) ACC HFA heatmap, electrodes x time
    pool = acc_trial.ravel()
    pool = pool[np.isfinite(pool)]
    vmax = float(np.nanpercentile(np.abs(pool), 98)) if pool.size else 1.0
    vmax = vmax if vmax > 0 else 1.0
    im = axes[1].imshow(
        acc_trial, aspect='auto', origin='lower',
        cmap='RdBu_r', vmin=-vmax, vmax=vmax,
        extent=[float(cue_times[0]), float(cue_times[-1]),
                -0.5, acc_trial.shape[0] - 0.5],
        interpolation='nearest',
    )
    axes[1].set_ylabel('ACC electrode')
    axes[1].set_title(f'ACC HFA heatmap ({acc_trial.shape[0]} electrodes x '
                      f'{acc_trial.shape[1]} time bins)')
    if kept_acc_labels.size <= 20:
        axes[1].set_yticks(np.arange(kept_acc_labels.size))
        axes[1].set_yticklabels([str(x) for x in kept_acc_labels], fontsize=7)
    else:
        step = max(1, int(np.ceil(kept_acc_labels.size / 20)))
        ticks = np.arange(0, kept_acc_labels.size, step)
        axes[1].set_yticks(ticks)
        axes[1].set_yticklabels([str(kept_acc_labels[i]) for i in ticks], fontsize=7)
    fig.colorbar(im, ax=axes[1], pad=0.01, label='HFA amplitude')

    # (3) ACC reactivation
    axes[2].plot(cue_times, example_react, '-o', color='black', markersize=3)
    axes[2].axhline(0, color='black', linestyle='--', alpha=0.35)
    axes[2].set_xlabel('cue time (s)')
    axes[2].set_ylabel('reactivation (ACC HFA . w_z)')
    axes[2].set_title("this trial's ACC reactivation trace "
                      f"(contrast={SELECTED_CONTRAST})")
    fig.tight_layout()
    plt.show()

    # ---- Plot B: phase vs reactivation scatter, pooled across (trial, time) ----
    fig, axes = plt.subplots(1, 2, figsize=(10.0, 4.0), sharey=True)
    for ax, cond, idx in [(axes[0], 'hit', hit_idx), (axes[1], 'miss', miss_idx)]:
        phi = phi_per_elec[idx, best_ei, :].reshape(-1)
        r = reactivation[idx, :].reshape(-1)
        mask = np.isfinite(phi) & np.isfinite(r)
        R, pref = weighted_rayleigh(phi[mask], r[mask], method='moore')
        ax.scatter(
            phi[mask], r[mask], s=12, alpha=0.35,
            color=('#3b82f6' if cond == 'hit' else '#ef4444'), edgecolor='none',
        )
        ax.axvline(pref, color='black', linestyle='--')
        ax.set_title(f'{cond}: {example_hpc_label}\n'
                     f'Moore R={R:.3f}, pref={np.degrees(pref):+.1f} deg')
        ax.set_xlabel('HPC theta phase (rad)')
        if cond == 'hit':
            ax.set_ylabel('reactivation')
    fig.suptitle(f'{DEMO_SUBJECT}: reactivation vs HPC theta phase '
                 f'(pooled trial x time samples) -- contrast={SELECTED_CONTRAST}')
    fig.tight_layout(rect=[0, 0, 1, 0.92])
    plt.show()

    # ---- Plot C: polar histogram of reactivation-weighted phase mass ----
    bins = np.linspace(-np.pi, np.pi, 13)
    centers_bin = (bins[:-1] + bins[1:]) / 2
    width = np.diff(bins)
    fig = plt.figure(figsize=(10.0, 4.5))
    for i, (cond, idx) in enumerate([('hit', hit_idx), ('miss', miss_idx)], start=1):
        ax = fig.add_subplot(1, 2, i, projection='polar')
        phi = phi_per_elec[idx, best_ei, :].reshape(-1)
        r = reactivation[idx, :].reshape(-1)
        mask = np.isfinite(phi) & np.isfinite(r)
        phi = phi[mask]
        r = r[mask]
        ranks = rankdata(np.abs(r), method='average')
        weights = ranks * np.sign(r)
        bin_idx = np.digitize(phi, bins) - 1
        bin_idx = np.clip(bin_idx, 0, bins.size - 2)
        bin_mass = np.array(
            [float(np.nansum(weights[bin_idx == bi])) for bi in range(bins.size - 1)],
            dtype=np.float64,
        )
        R, pref = weighted_rayleigh(phi, r, method='moore')
        pos = np.maximum(bin_mass, 0)
        neg = np.maximum(-bin_mass, 0)
        ax.bar(centers_bin, pos, width=width, bottom=0, alpha=0.75,
               color=('#3b82f6' if cond == 'hit' else '#ef4444'),
               label='positive mass')
        if np.any(neg > 0):
            ax.bar(centers_bin, neg, width=width, bottom=0, alpha=0.25,
                   color='gray', label='negative mass (abs)')
        rmax = float(np.nanmax(np.abs(bin_mass))) if np.nanmax(np.abs(bin_mass)) > 0 else 1.0
        ax.plot([pref, pref], [0, rmax], color='black', lw=2.5,
                label=f'pref={np.degrees(pref):+.1f} deg')
        ax.set_title(f'{cond}: weighted phase mass\nR={R:.3f}', fontsize=10)
        ax.legend(loc='upper right', bbox_to_anchor=(1.25, 1.15), fontsize=8)
    fig.suptitle(f'Polar view of reactivation-weighted phase mass '
                 f'(contrast={SELECTED_CONTRAST})', fontsize=12)
    fig.tight_layout(rect=[0, 0, 1, 0.92])
    plt.show()
else:
    print('Demo result not OK or hit condition not OK; skipping deep-dive plots.')


## Group loop

Run the runner over every subject with Haufe patterns for the current contrast. Per-subject NPZs
are cached in `analysis_outputs/acc_hpc_phase_rsa/<SELECTED_CONTRAST>/<split_tag>/per_subject/` and
auto-invalidated if the `RSA_PRIMITIVE` / `TRIAL_SPLIT` schema markers are missing or stale.

In [ ]:
# -----------------------------
# Group loop
# -----------------------------
RUN_OVERWRITE = False

results: List[Dict[str, Any]] = []
for sid in tqdm(subjects_with_patterns, desc='subjects'):
    try:
        results.append(run_subject_phase_rsa(sid, overwrite=RUN_OVERWRITE))
    except Exception as exc:
        results.append({'subject': sid, 'status': 'fail', 'reason': f'unhandled: {exc!r}'})

summary_df = pd.DataFrame([
    {'subject': r['subject'],
     'status': r['status'],
     'reason': r.get('reason', '')}
    for r in results
])
print('Status counts:')
print(summary_df['status'].value_counts())
print()
print('Skipped/failed subjects:')
print(
    summary_df.query("status not in ('ok', 'cached')")
    [['subject', 'status', 'reason']]
    .to_string(index=False)
)


## Group statistics -- phase organization

For **`TRIAL_SPLIT="hit_miss"`**: one-sample tests of hit / miss subject-level `mean_z` and `best_z`
vs 0, plus paired hit vs miss Wilcoxon across subjects.

For **`TRIAL_SPLIT="rpe_cat"`**: same per-subject summaries for `neg`, `neutral`, `pos`, then
**Friedman** test across the three matched subject-level vectors and **pairwise** signed-rank tests
(`neg` vs `neutral`, `neg` vs `pos`, `neutral` vs `pos`) using only subjects with finite values for
both members of each pair (raw p-values printed; CSV includes Holm-adjusted p where applicable).


In [ ]:
# -----------------------------
# Aggregate per-subject mean / best z across HPC electrodes
# -----------------------------
agg_rows = []
for sid in subjects_with_patterns:
    d = load_subject_npz(sid)
    if d is None:
        continue
    row = {'subject': sid}
    for cond in CONDITION_ORDER:
        status = str(d.get(f'{cond}_status', 'skip'))
        if status != 'ok':
            row[f'{cond}_mean_z'] = np.nan
            row[f'{cond}_best_z'] = np.nan
            row[f'{cond}_mean_rho_obs'] = np.nan
            row[f'{cond}_n_trials'] = 0
            row[f'{cond}_n_elec'] = 0
            continue
        z_arr = np.asarray(d[f'{cond}_z'], dtype=np.float64)
        rho_arr = np.asarray(d[f'{cond}_rho_obs'], dtype=np.float64)
        finite = np.isfinite(z_arr)
        row[f'{cond}_mean_z'] = float(np.nanmean(z_arr[finite])) if finite.any() else np.nan
        row[f'{cond}_best_z'] = float(np.nanmax(z_arr[finite])) if finite.any() else np.nan
        row[f'{cond}_mean_rho_obs'] = float(np.nanmean(rho_arr[finite])) if finite.any() else np.nan
        row[f'{cond}_n_trials'] = int(d.get(f'{cond}_n_trials', 0))
        row[f'{cond}_n_elec'] = int(finite.sum())
    agg_rows.append(row)

agg_df = pd.DataFrame(agg_rows)
agg_df.to_csv(analysis_output_root / 'group_subject_summary.csv', index=False)
print(f'Subjects with any cached output: {len(agg_df)}')
print(agg_df.head().to_string(index=False))


def _paired_summary(values: np.ndarray, label: str) -> Dict[str, float]:
    v = np.asarray(values, dtype=np.float64)
    v = v[np.isfinite(v)]
    out = {'label': label, 'n': int(v.size), 'mean': float(np.mean(v)) if v.size else np.nan,
           'sem': float(np.std(v, ddof=1) / np.sqrt(v.size)) if v.size > 1 else np.nan,
           't_stat': np.nan, 't_p': np.nan, 'w_stat': np.nan, 'w_p': np.nan}
    if v.size > 2:
        t, p = ttest_1samp(v, popmean=0.0)
        out['t_stat'] = float(t)
        out['t_p'] = float(p)
        try:
            w, wp = wilcoxon(v)
            out['w_stat'] = float(w)
            out['w_p'] = float(wp)
        except Exception:
            pass
    return out


group_stats_rows: List[Dict[str, Any]] = []

if TRIAL_SPLIT == 'hit_miss':
    for cond in ('hit', 'miss'):
        for key in ('mean_z', 'best_z'):
            group_stats_rows.append(_paired_summary(agg_df[f'{cond}_{key}'].to_numpy(),
                                                     f'{cond}_{key}'))

    def _paired_hit_miss(arr_hit: np.ndarray, arr_miss: np.ndarray, label: str) -> Dict[str, Any]:
        h = np.asarray(arr_hit, dtype=np.float64)
        m = np.asarray(arr_miss, dtype=np.float64)
        mask = np.isfinite(h) & np.isfinite(m)
        h = h[mask]
        m = m[mask]
        out = {'label': label, 'n': int(mask.sum()),
               'mean_diff': float(np.mean(h - m)) if mask.any() else np.nan,
               't_stat': np.nan, 't_p': np.nan, 'w_stat': np.nan, 'w_p': np.nan}
        if mask.sum() > 2:
            t, p = ttest_rel(h, m)
            out['t_stat'] = float(t)
            out['t_p'] = float(p)
            try:
                w, wp = wilcoxon(h, m)
                out['w_stat'] = float(w)
                out['w_p'] = float(wp)
            except Exception:
                pass
        return out

    for key in ('mean_z', 'best_z'):
        group_stats_rows.append(_paired_hit_miss(
            agg_df[f'hit_{key}'].to_numpy(),
            agg_df[f'miss_{key}'].to_numpy(),
            label=f'hit_vs_miss_{key}',
        ))

    group_stats_df = pd.DataFrame(group_stats_rows)
    group_stats_df.to_csv(analysis_output_root / 'group_paired_stats.csv', index=False)
    print()
    print('Group statistics (hit/miss):')
    print(group_stats_df.to_string(index=False))

elif TRIAL_SPLIT == 'rpe_cat':
    cats = ('neg', 'neutral', 'pos')

    def _holm_adjust(pvals: List[float]) -> List[float]:
        p = np.asarray(pvals, dtype=np.float64)
        m = int(p.size)
        if m == 0:
            return []
        order = np.argsort(p)
        adj = np.empty_like(p)
        running_max = 0.0
        for rank in range(m, 0, -1):
            i = int(order[rank - 1])
            val = float(p[i] * m / rank)
            running_max = max(running_max, val)
            adj[i] = min(running_max, 1.0)
        return [float(x) for x in adj]

    rpe_rows: List[Dict[str, Any]] = []
    for metric in ('mean_z', 'best_z'):
        cols = [f'{c}_{metric}' for c in cats]
        mat = agg_df[cols].to_numpy(dtype=np.float64)
        mask = np.all(np.isfinite(mat), axis=1)
        n_sub = int(mask.sum())
        row: Dict[str, Any] = {
            'test': 'friedman', 'metric': metric, 'n_subjects': n_sub,
            'statistic': np.nan, 'p': np.nan, 'p_holm': np.nan,
        }
        if n_sub >= 3:
            stat_f, p_f = friedmanchisquare(
                mat[mask, 0], mat[mask, 1], mat[mask, 2],
            )
            row['statistic'] = float(stat_f)
            row['p'] = float(p_f)
        rpe_rows.append(row)
        print(
            f'Friedman ({metric}): n_subjects={n_sub}  '
            f'statistic={row["statistic"]}  p={row["p"]}'
        )

        raw_ps: List[float] = []
        pair_rows: List[Dict[str, Any]] = []
        for i in range(3):
            for j in range(i + 1, 3):
                a, b = cats[i], cats[j]
                va = agg_df[f'{a}_{metric}'].to_numpy(dtype=np.float64)
                vb = agg_df[f'{b}_{metric}'].to_numpy(dtype=np.float64)
                m2 = np.isfinite(va) & np.isfinite(vb)
                pr = np.nan
                wst = np.nan
                if int(m2.sum()) >= 3:
                    try:
                        wst, pr = wilcoxon(va[m2], vb[m2], alternative='two-sided')
                    except ValueError:
                        pass
                pr_f = float(pr) if np.isfinite(pr) else np.nan
                raw_ps.append(pr_f)
                pair_rows.append({
                    'test': 'wilcoxon_pairwise', 'metric': metric,
                    'contrast': f'{a}_vs_{b}', 'n_subjects': int(m2.sum()),
                    'statistic': float(wst) if np.isfinite(wst) else np.nan,
                    'p': pr_f,
                    'p_holm': np.nan,
                })
                print(
                    f'  Wilcoxon {a} vs {b} ({metric}): n={int(m2.sum())}  '
                    f'statistic={wst}  p={pr}'
                )
        finite_ps = [x for x in raw_ps if np.isfinite(x)]
        holm = _holm_adjust(finite_ps) if finite_ps else []
        hi = 0
        for pr, prow in zip(raw_ps, pair_rows):
            if np.isfinite(pr):
                prow['p_holm'] = holm[hi]
                hi += 1
            rpe_rows.append(prow)

    rpe_stats_df = pd.DataFrame(rpe_rows)
    rpe_stats_df.to_csv(analysis_output_root / 'group_rpe_cat_stats.csv', index=False)
    print()
    print('Wrote', analysis_output_root / 'group_rpe_cat_stats.csv')
else:
    raise ValueError(f'Unknown TRIAL_SPLIT: {TRIAL_SPLIT!r}')


## Figures

Two summary figures when `TRIAL_SPLIT="hit_miss"`: paired strip plots and per-electrode distributions.

When `TRIAL_SPLIT="rpe_cat"`: three-level strip / violin summaries for `neg` / `neutral` / `pos`, plus
a subject-level `pos - neg` mean-z panel.


In [ ]:
# -----------------------------
# Summary figures
# -----------------------------
if len(agg_df) == 0:
    print('agg_df empty; figures skipped.')
elif TRIAL_SPLIT == 'hit_miss':
    # Figure 1: subject-level hit vs miss strip
    fig, axes = plt.subplots(1, 2, figsize=(9.0, 4.5))
    for ax, key in zip(axes, ('mean_z', 'best_z')):
        plot_df = agg_df[['subject', f'hit_{key}', f'miss_{key}']].copy()
        plot_df = plot_df.rename(columns={f'hit_{key}': 'hit', f'miss_{key}': 'miss'})
        long = plot_df.melt(id_vars='subject', var_name='cond', value_name='z')
        sns.stripplot(data=long, x='cond', y='z', hue='cond', ax=ax, jitter=0.05,
                      palette={'hit': '#3b82f6', 'miss': '#ef4444'}, size=6, legend=False)
        for _, row in plot_df.dropna().iterrows():
            ax.plot([0, 1], [row['hit'], row['miss']],
                    color='gray', alpha=0.4, lw=0.8)
        ax.axhline(0, color='black', linestyle='--', alpha=0.4)
        ax.set_ylabel(f'{key} (HPC)')
        ax.set_xlabel('')
        ax.set_title(f'{key} per subject')
    fig.suptitle(
        f'Subject-level phase-RSA z (hit/miss) — contrast={SELECTED_CONTRAST}',
        fontsize=12,
    )
    fig.tight_layout(rect=[0, 0, 1, 0.93])
    fig.savefig(figure_output_root / 'fig1_subject_hit_vs_miss.png', dpi=200,
                bbox_inches='tight')
    plt.show()

    # Figure 2: per-electrode z distribution across subjects
    elec_rows = []
    for sid in subjects_with_patterns:
        d = load_subject_npz(sid)
        if d is None:
            continue
        hpc_labels = np.asarray(d['hpc_electrode_labels'], dtype=object)
        for cond in ('hit', 'miss'):
            if str(d.get(f'{cond}_status', 'skip')) != 'ok':
                continue
            z_arr = np.asarray(d[f'{cond}_z'], dtype=np.float64)
            rho_arr = np.asarray(d[f'{cond}_rho_obs'], dtype=np.float64)
            p_arr = np.asarray(d[f'{cond}_p'], dtype=np.float64)
            for ei, lbl in enumerate(hpc_labels):
                elec_rows.append({
                    'subject': sid,
                    'cond': cond,
                    'electrode': str(lbl),
                    'z': float(z_arr[ei]),
                    'rho_obs': float(rho_arr[ei]),
                    'p': float(p_arr[ei]),
                })

    elec_df = pd.DataFrame(elec_rows)
    elec_df.to_csv(analysis_output_root / 'per_electrode_long.csv', index=False)

    if len(elec_df) > 0:
        fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.6))
        sns.violinplot(data=elec_df, x='cond', y='z', hue='cond', ax=axes[0],
                       palette={'hit': '#3b82f6', 'miss': '#ef4444'},
                       inner='quartile', cut=0, legend=False)
        sns.stripplot(data=elec_df, x='cond', y='z', hue='cond', ax=axes[0],
                      palette={'hit': '#0b3a8b', 'miss': '#8b0b0b'},
                      size=3, alpha=0.45, jitter=0.18, legend=False)
        axes[0].axhline(0, color='black', linestyle='--', alpha=0.4)
        axes[0].set_title('Per-electrode z across all subjects')
        axes[0].set_ylabel('z (phase-similarity RSA)')

        pivot = elec_df.pivot_table(
            index='subject', columns='cond', values='z', aggfunc='mean'
        )
        if {'hit', 'miss'}.issubset(pivot.columns):
            diff = (pivot['hit'] - pivot['miss']).dropna().sort_values()
            axes[1].barh(np.arange(diff.size), diff.values,
                         color=['#3b82f6' if v >= 0 else '#ef4444' for v in diff.values])
            axes[1].set_yticks(np.arange(diff.size))
            axes[1].set_yticklabels(diff.index.tolist(), fontsize=8)
            axes[1].axvline(0, color='black', linestyle='--', alpha=0.5)
            axes[1].set_xlabel('hit - miss subject mean z')
            axes[1].set_title('Hit minus miss subject mean z')
        fig.suptitle(f'HPC phase-RSA per-electrode summary — contrast={SELECTED_CONTRAST}')
        fig.tight_layout(rect=[0, 0, 1, 0.93])
        fig.savefig(figure_output_root / 'fig2_per_electrode_summary.png', dpi=200,
                    bbox_inches='tight')
        plt.show()
    else:
        print('elec_df empty; figure 2 skipped.')

else:
    cats = ['neg', 'neutral', 'pos']
    palette = {'neg': '#ef4444', 'neutral': '#94a3b8', 'pos': '#22c55e'}
    # Figure 1: three-condition strip + within-subject lines
    fig, axes = plt.subplots(1, 2, figsize=(10.0, 4.8))
    for ax, key in zip(axes, ('mean_z', 'best_z')):
        cols = [f'{c}_{key}' for c in cats]
        plot_df = agg_df[['subject'] + cols].copy()
        long = plot_df.melt(id_vars='subject', var_name='cond', value_name='z')
        sns.stripplot(
            data=long, x='cond', y='z', hue='cond', ax=ax, jitter=0.08,
            order=cats, hue_order=cats, palette=palette, size=6, legend=False,
        )
        for _, row in plot_df.iterrows():
            vals = [row[c] for c in cols]
            if all(np.isfinite(v) for v in vals):
                ax.plot([0, 1, 2], vals, color='gray', alpha=0.35, lw=0.9)
        ax.axhline(0, color='black', linestyle='--', alpha=0.4)
        ax.set_ylabel(f'{key} (HPC)')
        ax.set_xlabel('rpe_cat')
        ax.set_title(f'{key} per subject')
    fig.suptitle(
        f'Subject-level phase-RSA z (rpe_cat) — contrast={SELECTED_CONTRAST}',
        fontsize=12,
    )
    fig.tight_layout(rect=[0, 0, 1, 0.93])
    fig.savefig(figure_output_root / 'fig1_subject_rpe_cat_strip.png', dpi=200,
                bbox_inches='tight')
    plt.show()

    # Figure 2: violin + pos minus neg
    elec_rows = []
    for sid in subjects_with_patterns:
        d = load_subject_npz(sid)
        if d is None:
            continue
        hpc_labels = np.asarray(d['hpc_electrode_labels'], dtype=object)
        for cond in cats:
            if str(d.get(f'{cond}_status', 'skip')) != 'ok':
                continue
            z_arr = np.asarray(d[f'{cond}_z'], dtype=np.float64)
            rho_arr = np.asarray(d[f'{cond}_rho_obs'], dtype=np.float64)
            p_arr = np.asarray(d[f'{cond}_p'], dtype=np.float64)
            for ei, lbl in enumerate(hpc_labels):
                elec_rows.append({
                    'subject': sid,
                    'cond': cond,
                    'electrode': str(lbl),
                    'z': float(z_arr[ei]),
                    'rho_obs': float(rho_arr[ei]),
                    'p': float(p_arr[ei]),
                })
    elec_df = pd.DataFrame(elec_rows)
    elec_df.to_csv(analysis_output_root / 'per_electrode_long.csv', index=False)

    if len(elec_df) > 0:
        fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.6))
        sns.violinplot(
            data=elec_df, x='cond', y='z', hue='cond', ax=axes[0],
            order=cats, hue_order=cats, palette=palette,
            inner='quartile', cut=0, legend=False,
        )
        sns.stripplot(
            data=elec_df, x='cond', y='z', hue='cond', ax=axes[0],
            order=cats, hue_order=cats,
            palette={'neg': '#7f1d1d', 'neutral': '#475569', 'pos': '#14532d'},
            size=3, alpha=0.45, jitter=0.18, legend=False,
        )
        axes[0].axhline(0, color='black', linestyle='--', alpha=0.4)
        axes[0].set_title('Per-electrode z across all subjects')
        axes[0].set_ylabel('z (phase-similarity RSA)')

        sub_mean = elec_df.groupby(['subject', 'cond'])['z'].mean().unstack('cond')
        if {'pos', 'neg'}.issubset(sub_mean.columns):
            diff = (sub_mean['pos'] - sub_mean['neg']).dropna().sort_values()
            axes[1].barh(np.arange(diff.size), diff.values,
                         color=['#22c55e' if v >= 0 else '#ef4444' for v in diff.values])
            axes[1].set_yticks(np.arange(diff.size))
            axes[1].set_yticklabels(diff.index.tolist(), fontsize=8)
            axes[1].axvline(0, color='black', linestyle='--', alpha=0.5)
            axes[1].set_xlabel('subject mean z: pos - neg')
            axes[1].set_title('Pos minus neg (subject mean z)')
        fig.suptitle(f'HPC phase-RSA (rpe_cat) — contrast={SELECTED_CONTRAST}')
        fig.tight_layout(rect=[0, 0, 1, 0.93])
        fig.savefig(figure_output_root / 'fig2_rpe_cat_per_electrode.png', dpi=200,
                    bbox_inches='tight')
        plt.show()
    else:
        print('elec_df empty; rpe_cat figure 2 skipped.')


## Supplementary Moore-Rayleigh preferred-phase diagnostic

Headline test (above) asks: *are reactivation values more similar at more similar phases?*
This diagnostic answers a complementary question: *at what phase is reactivation high?* For each
subject we recompute the Moore-Rayleigh resultant (`R`, preferred phase) on the best-z HPC
electrode using pooled (trial x time) samples and plot the distribution of preferred phases.

In [ ]:
# -----------------------------
# Preferred-phase diagnostic (best-z HPC electrode per subject)
# -----------------------------
diag_rows = []
for sid in subjects_with_patterns:
    d = load_subject_npz(sid)
    if d is None:
        continue
    cue_times = np.asarray(d['cue_times'], dtype=np.float64)
    reactivation = np.asarray(d['reactivation'], dtype=np.float64)
    phi_per_elec = np.asarray(d['phi_per_elec'], dtype=np.float64)
    hpc_labels = np.asarray(d['hpc_electrode_labels'], dtype=object)
    conds = npz_condition_order(d)
    cond_idx = {
        c: np.asarray(d[f'{c}_indices'], dtype=np.int64)
        for c in conds if f'{c}_indices' in d
    }
    for cond in conds:
        if cond not in cond_idx:
            continue
        if str(d.get(f'{cond}_status', 'skip')) != 'ok':
            continue
        z_arr = np.asarray(d[f'{cond}_z'], dtype=np.float64)
        if not np.any(np.isfinite(z_arr)):
            continue
        ei_best = int(np.nanargmax(z_arr))
        idx = cond_idx[cond]
        phi = phi_per_elec[idx, ei_best, :].reshape(-1)
        r = reactivation[idx, :].reshape(-1)
        mask = np.isfinite(phi) & np.isfinite(r)
        if mask.sum() < 50:
            continue
        R, pref = weighted_rayleigh(phi[mask], r[mask], method='moore')
        diag_rows.append({
            'subject': sid, 'cond': cond,
            'electrode': str(hpc_labels[ei_best]),
            'R': float(R), 'pref_rad': float(pref),
            'pref_deg': float(np.degrees(pref)),
        })

diag_df = pd.DataFrame(diag_rows)
diag_df.to_csv(analysis_output_root / 'preferred_phase_diagnostic.csv', index=False)

if len(diag_df) > 0:
    palette_diag = {
        'hit': '#3b82f6', 'miss': '#ef4444',
        'neg': '#ef4444', 'neutral': '#94a3b8', 'pos': '#22c55e',
    }
    conds_plot = sorted(diag_df['cond'].unique().tolist())
    n_p = len(conds_plot)
    fig = plt.figure(figsize=(4.5 * n_p, 4.5))
    for i, cond in enumerate(conds_plot, start=1):
        ax = fig.add_subplot(1, n_p, i, projection='polar')
        sub = diag_df[diag_df['cond'] == cond]
        if len(sub) == 0:
            ax.set_title(f'{cond}: no data', fontsize=10)
            continue
        for _, row in sub.iterrows():
            ax.plot([row['pref_rad'], row['pref_rad']], [0, row['R']],
                    color=palette_diag.get(cond, '#334155'),
                    alpha=0.7, lw=1.5)
        Rg, prefg = weighted_rayleigh(
            sub['pref_rad'].to_numpy(),
            np.ones(len(sub), dtype=np.float64),
            method='standard',
        )
        rmax = float(sub['R'].max()) if len(sub) else 1.0
        if np.isfinite(prefg):
            ax.plot([prefg, prefg], [0, rmax], color='black', lw=2.5,
                    label=f'group pref={np.degrees(prefg):+.1f} deg (R={Rg:.2f})')
            ax.legend(loc='upper right', bbox_to_anchor=(1.30, 1.15), fontsize=8)
        ax.set_title(f'{cond}: subject preferred phases (best HPC elec)\n'
                     f'n={len(sub)} subjects', fontsize=10)
    fig.suptitle(
        f'Preferred phase diagnostic — contrast={SELECTED_CONTRAST} — {TRIAL_SPLIT}',
        fontsize=12,
    )
    fig.tight_layout(rect=[0, 0, 1, 0.92])
    fig.savefig(figure_output_root / 'fig3_preferred_phase_diagnostic.png', dpi=200,
                bbox_inches='tight')
    plt.show()
else:
    print('diag_df empty; preferred-phase diagnostic figure skipped.')
